# 02 — Preprocessing v3 (Per-User Normalization + Feature Engineering)

Key changes from v2:
1. **Per-user normalization** — removes user-specific sensor bias before windowing
2. **Magnitude features** — adds rotation-invariant accel/gyro magnitude (8 channels total)
3. **All 11 activities** with 3x augmentation

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [2]:
DATA_DIR = Path('../data')
RAW_PATH = DATA_DIR / 'raw' / 'AdamSense.csv'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Raw sensor columns (wrist IMU)
RAW_SENSOR_COLS = ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w']
# We'll add 2 magnitude features -> 8 total
ALL_SENSOR_COLS = RAW_SENSOR_COLS + ['Acc_mag', 'Gyro_mag']
NUM_FEATURES = len(ALL_SENSOR_COLS)

# Windowing
SAMPLING_RATE = 50
WINDOW_SIZE = 128
OVERLAP = 0.5
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))
LABEL_THRESHOLD = 0.75

# Train/test split by user
TEST_USERS = [1, 2]

print(f'Window: {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLING_RATE:.2f}s)')
print(f'Step: {STEP_SIZE} samples')
print(f'Features: {NUM_FEATURES} — {ALL_SENSOR_COLS}')

Window: 128 samples (2.56s)
Step: 64 samples
Features: 8 — ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w', 'Acc_mag', 'Gyro_mag']


## 2. Load Data

In [3]:
df = pd.read_csv(RAW_PATH)
print(f'Total samples: {len(df):,}')
print(f'Users: {sorted(df["User"].unique())}')
print(f'\nActivities ({df["Activity"].nunique()}):')
for act, count in df['Activity'].value_counts().items():
    print(f'  {act:25s} {count:>8,}')

Total samples: 709,582
Users: [2, 3, 5, 6, 7, 8, 10, 14, 15, 16]

Activities (11):
  nape_rubbing                67,859
  knuckles_cracking           67,498
  smoking                     66,937
  ear_rubbing                 66,599
  hair_pulling                65,973
  forehead_rubbing            64,604
  hand_scratching             64,490
  nail_biting                 63,457
  hand_tapping                63,220
  sitting                     59,996
  standing                    58,949


## 3. Per-User Normalization

Each user wears the sensor differently and has different body mechanics.  
Normalizing per-user removes this bias so the model learns **activity patterns**, not **user identity**.

In [4]:
# Compute per-user statistics on raw sensor columns
print('Per-user sensor statistics (before normalization):')
print(f'{"User":>6} {"Ax_mean":>10} {"Ay_mean":>10} {"Az_mean":>10} {"Gx_mean":>10}')
print('-' * 50)

user_stats = {}
for user in sorted(df['User'].unique()):
    user_data = df.loc[df['User'] == user, RAW_SENSOR_COLS]
    mean = user_data.mean().values
    std = user_data.std().values
    std[std < 1e-8] = 1.0
    user_stats[user] = {'mean': mean, 'std': std}
    print(f'{user:>6} {mean[0]:>10.4f} {mean[1]:>10.4f} {mean[2]:>10.4f} {mean[3]:>10.4f}')

# Apply per-user normalization
df_norm = df.copy()
for user, stats in user_stats.items():
    mask = df_norm['User'] == user
    df_norm.loc[mask, RAW_SENSOR_COLS] = (
        (df_norm.loc[mask, RAW_SENSOR_COLS].values - stats['mean']) / stats['std']
    )

print('\nPer-user normalization applied (each user now has ~zero mean, ~unit std)')

Per-user sensor statistics (before normalization):
  User    Ax_mean    Ay_mean    Az_mean    Gx_mean
--------------------------------------------------
     2     0.4177    -0.1233    -0.3678    -0.8293
     3     0.4731    -0.0767    -0.3560    -1.0009
     5     0.3914    -0.0983    -0.2733    -1.0323
     6     0.1052     0.2598    -0.5705    -0.8432
     7     0.3508    -0.1776    -0.4585    -0.6291
     8     0.3502    -0.0753    -0.4541    -0.6535
    10     0.3678    -0.1345    -0.4530    -1.1524
    14     0.3889     0.0843    -0.3076    -0.7841
    15     0.1119     0.3819    -0.4664    -1.0234
    16    -0.1973     0.2807    -0.6387    -1.0336

Per-user normalization applied (each user now has ~zero mean, ~unit std)


## 4. Add Magnitude Features

Magnitude = sqrt(x^2 + y^2 + z^2) is **rotation-invariant**.  
It captures *how much* motion there is regardless of sensor orientation — which varies across users.

In [5]:
# Compute after per-user normalization
df_norm['Acc_mag'] = np.sqrt(
    df_norm['Ax_w']**2 + df_norm['Ay_w']**2 + df_norm['Az_w']**2
)
df_norm['Gyro_mag'] = np.sqrt(
    df_norm['Gx_w']**2 + df_norm['Gy_w']**2 + df_norm['Gz_w']**2
)

print(f'Added magnitude features:')
print(f'  Acc_mag  — mean: {df_norm["Acc_mag"].mean():.4f}, std: {df_norm["Acc_mag"].std():.4f}')
print(f'  Gyro_mag — mean: {df_norm["Gyro_mag"].mean():.4f}, std: {df_norm["Gyro_mag"].std():.4f}')
print(f'\nFinal feature set ({NUM_FEATURES}): {ALL_SENSOR_COLS}')

Added magnitude features:
  Acc_mag  — mean: 1.6332, std: 0.5767
  Gyro_mag — mean: 1.0924, std: 1.3441

Final feature set (8): ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w', 'Acc_mag', 'Gyro_mag']


## 5. Label Mapping (All 11 Activities)

In [6]:
ALL_ACTIVITIES = sorted(df_norm['Activity'].unique().tolist())
NUM_CLASSES = len(ALL_ACTIVITIES)
LABEL_MAP = {act: i for i, act in enumerate(ALL_ACTIVITIES)}

df_norm['label'] = df_norm['Activity'].map(LABEL_MAP)

print(f'All {NUM_CLASSES} activities:')
for act, idx in LABEL_MAP.items():
    print(f'  {idx:2d}: {act}')

All 11 activities:
   0: ear_rubbing
   1: forehead_rubbing
   2: hair_pulling
   3: hand_scratching
   4: hand_tapping
   5: knuckles_cracking
   6: nail_biting
   7: nape_rubbing
   8: sitting
   9: smoking
  10: standing


np.save(PROCESSED_DIR / 'X_train.npy', X_train_aug)
np.save(PROCESSED_DIR / 'y_train.npy', y_train_aug)
np.save(PROCESSED_DIR / 'X_test.npy', X_test)
np.save(PROCESSED_DIR / 'y_test.npy', y_test)
np.save(PROCESSED_DIR / 'X_train_original.npy', X_train)
np.save(PROCESSED_DIR / 'y_train_original.npy', y_train)

# Save all windows + user labels for LOUO cross-validation
np.save(PROCESSED_DIR / 'X_all.npy', (X_all - sensor_mean) / sensor_std)  # normalized
np.save(PROCESSED_DIR / 'y_all.npy', y_all)
np.save(PROCESSED_DIR / 'users_all.npy', users_all)

preprocessing_info = {
    'sensor_columns': ALL_SENSOR_COLS,
    'raw_sensor_columns': RAW_SENSOR_COLS,
    'num_features': NUM_FEATURES,
    'num_classes': NUM_CLASSES,
    'activities': ALL_ACTIVITIES,
    'label_map': LABEL_MAP,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'overlap': OVERLAP,
    'sampling_rate': SAMPLING_RATE,
    'sensor_mean': sensor_mean,
    'sensor_std': sensor_std,
    'user_stats': user_stats,
    'test_users': TEST_USERS,
    'train_samples_original': len(X_train),
    'train_samples_augmented': len(X_train_aug),
    'test_samples': len(X_test),
    'label_threshold': LABEL_THRESHOLD,
    'num_aug_copies': NUM_AUG_COPIES,
    'per_user_normalization': True,
    'magnitude_features': True,
}

with open(PROCESSED_DIR / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print('Saved files:')
for p in sorted(list(PROCESSED_DIR.glob('*.npy')) + list(PROCESSED_DIR.glob('*.pkl'))):
    size = p.stat().st_size
    unit = 'KB' if size < 1e6 else 'MB'
    val = size / 1024 if size < 1e6 else size / (1024*1024)
    print(f'  {p.name:35s} {val:>8.1f} {unit}')

In [7]:
def extract_windows(group_df, window_size, step_size, sensor_cols, label_threshold):
    """Extract fixed-size windows with majority label voting."""
    sensor_data = group_df[sensor_cols].values
    labels = group_df['label'].values
    n_samples = len(sensor_data)
    
    windows, window_labels = [], []
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window_lab = labels[start:end]
        counts = Counter(window_lab)
        dominant_label, dominant_count = counts.most_common(1)[0]
        if dominant_count / window_size >= label_threshold:
            windows.append(sensor_data[start:end])
            window_labels.append(dominant_label)
    
    return windows, window_labels


all_windows, all_labels, all_users = [], [], []

for user in sorted(df_norm['User'].unique()):
    user_wins = 0
    for activity in ALL_ACTIVITIES:
        mask = (df_norm['User'] == user) & (df_norm['Activity'] == activity)
        segment = df_norm.loc[mask]
        if len(segment) < WINDOW_SIZE:
            continue
        wins, labs = extract_windows(segment, WINDOW_SIZE, STEP_SIZE, ALL_SENSOR_COLS, LABEL_THRESHOLD)
        all_windows.extend(wins)
        all_labels.extend(labs)
        all_users.extend([user] * len(wins))
        user_wins += len(wins)
    print(f'  User {user:>2d}: {user_wins:>4d} windows')

X_all = np.array(all_windows, dtype=np.float32)
y_all = np.array(all_labels, dtype=np.int32)
users_all = np.array(all_users)

print(f'\nTotal windows: {len(X_all)}')
print(f'Shape: {X_all.shape}')

  User  2: 1018 windows
  User  3:  934 windows
  User  5:  804 windows
  User  6:  945 windows
  User  7: 1054 windows
  User  8:  936 windows
  User 10: 1065 windows
  User 14: 1036 windows
  User 15: 1020 windows
  User 16: 2115 windows

Total windows: 10927
Shape: (10927, 128, 8)


## 7. Train/Test Split

In [8]:
test_mask = np.isin(users_all, TEST_USERS)
train_mask = ~test_mask

X_train_raw = X_all[train_mask]
y_train = y_all[train_mask]
users_train = users_all[train_mask]
X_test_raw = X_all[test_mask]
y_test = y_all[test_mask]

print(f'Train: {X_train_raw.shape[0]} windows (users {sorted(set(users_all[train_mask]))})')
print(f'Test:  {X_test_raw.shape[0]} windows (users {sorted(set(users_all[test_mask]))})')

print(f'\nTrain class distribution:')
for idx, act in enumerate(ALL_ACTIVITIES):
    c = np.sum(y_train == idx)
    print(f'  {idx:2d}: {act:25s} {c:>5d}')

print(f'\nTest class distribution:')
for idx, act in enumerate(ALL_ACTIVITIES):
    c = np.sum(y_test == idx)
    print(f'  {idx:2d}: {act:25s} {c:>5d}')

Train: 9909 windows (users [3, 5, 6, 7, 8, 10, 14, 15, 16])
Test:  1018 windows (users [2])

Train class distribution:
   0: ear_rubbing                 926
   1: forehead_rubbing            896
   2: hair_pulling                906
   3: hand_scratching             897
   4: hand_tapping                895
   5: knuckles_cracking           918
   6: nail_biting                 896
   7: nape_rubbing                947
   8: sitting                     856
   9: smoking                     932
  10: standing                    840

Test class distribution:
   0: ear_rubbing                 101
   1: forehead_rubbing             99
   2: hair_pulling                111
   3: hand_scratching              96
   4: hand_tapping                 77
   5: knuckles_cracking           122
   6: nail_biting                  81
   7: nape_rubbing                 98
   8: sitting                      67
   9: smoking                     100
  10: standing                     66


## 8. Global Z-score Normalization

After per-user normalization, apply global z-score so all features are on the same scale.  
Stats computed from training data only.

In [9]:
train_flat = X_train_raw.reshape(-1, NUM_FEATURES)
sensor_mean = train_flat.mean(axis=0)
sensor_std = train_flat.std(axis=0)
sensor_std[sensor_std < 1e-8] = 1.0

print('Global normalization parameters (post per-user norm):')
for i, col in enumerate(ALL_SENSOR_COLS):
    print(f'  {col:10s}: mean={sensor_mean[i]:>8.4f}, std={sensor_std[i]:>8.4f}')

X_train = (X_train_raw - sensor_mean) / sensor_std
X_test = (X_test_raw - sensor_mean) / sensor_std

print(f'\nX_train range: [{X_train.min():.2f}, {X_train.max():.2f}]')
print(f'X_test  range: [{X_test.min():.2f}, {X_test.max():.2f}]')

Global normalization parameters (post per-user norm):
  Ax_w      : mean=  0.0010, std=  1.0004
  Ay_w      : mean=  0.0013, std=  1.0006
  Az_w      : mean=  0.0007, std=  0.9999
  Gx_w      : mean= -0.0000, std=  0.9988
  Gy_w      : mean= -0.0003, std=  0.9990
  Gz_w      : mean= -0.0001, std=  0.9977
  Acc_mag   : mean=  1.6381, std=  0.5640
  Gyro_mag  : mean=  1.0868, std=  1.3462

X_train range: [-15.50, 29.50]
X_test  range: [-17.04, 19.19]


## 9. Data Augmentation

In [10]:
def augment_jitter(x, sigma=0.05):
    return x + np.random.normal(0, sigma, x.shape).astype(np.float32)

def augment_scaling(x, sigma=0.1):
    factors = np.random.normal(1.0, sigma, (1, x.shape[1])).astype(np.float32)
    return x * factors

def augment_time_warp(x, sigma=0.2, num_knots=4):
    from scipy.interpolate import CubicSpline
    orig_steps = np.arange(x.shape[0])
    knot_positions = np.linspace(0, x.shape[0] - 1, num_knots + 2)
    knot_values = knot_positions + np.random.normal(0, sigma, len(knot_positions)).cumsum()
    knot_values = np.clip(knot_values, 0, x.shape[0] - 1)
    knot_values[0] = 0
    knot_values[-1] = x.shape[0] - 1
    cs = CubicSpline(knot_positions, knot_values)
    warped_steps = np.clip(cs(orig_steps), 0, x.shape[0] - 1)
    result = np.zeros_like(x)
    for c in range(x.shape[1]):
        result[:, c] = np.interp(warped_steps, orig_steps, x[:, c])
    return result

def augment_rotation(x):
    """Small random 3D rotation applied to accel and gyro independently."""
    def random_rotation_matrix(max_angle=15):
        angles = np.radians(np.random.uniform(-max_angle, max_angle, 3))
        cx, cy, cz = np.cos(angles)
        sx, sy, sz = np.sin(angles)
        Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
        Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
        Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
        return Rz @ Ry @ Rx
    
    result = x.copy()
    R_acc = random_rotation_matrix()
    result[:, :3] = (R_acc @ result[:, :3].T).T
    R_gyro = random_rotation_matrix()
    result[:, 3:6] = (R_gyro @ result[:, 3:6].T).T
    # Recompute magnitudes after rotation (they should be ~same but let's be exact)
    result[:, 6] = np.sqrt(result[:, 0]**2 + result[:, 1]**2 + result[:, 2]**2)
    result[:, 7] = np.sqrt(result[:, 3]**2 + result[:, 4]**2 + result[:, 5]**2)
    return result.astype(np.float32)

def augment_magnitude_warp(x, sigma=0.2, num_knots=4):
    from scipy.interpolate import CubicSpline
    orig_steps = np.arange(x.shape[0])
    knot_positions = np.linspace(0, x.shape[0] - 1, num_knots + 2)
    knot_values = np.random.normal(1.0, sigma, len(knot_positions))
    cs = CubicSpline(knot_positions, knot_values)
    warp = cs(orig_steps).astype(np.float32)
    return x * warp[:, np.newaxis]

def augment_permutation(x, n_segments=5):
    seg_len = x.shape[0] // n_segments
    segments = [x[i*seg_len:(i+1)*seg_len] for i in range(n_segments)]
    remainder = x[n_segments*seg_len:]
    np.random.shuffle(segments)
    if len(remainder) > 0:
        segments.append(remainder)
    return np.concatenate(segments, axis=0)[:x.shape[0]]


def apply_augmentation(x, p=0.5):
    x = x.copy()
    if np.random.random() < p:
        x = augment_jitter(x, sigma=0.05)
    if np.random.random() < p:
        x = augment_scaling(x, sigma=0.1)
    if np.random.random() < p:
        x = augment_rotation(x)
    if np.random.random() < 0.3:
        x = augment_time_warp(x, sigma=0.2)
    if np.random.random() < 0.3:
        x = augment_magnitude_warp(x, sigma=0.2)
    if np.random.random() < 0.2:
        x = augment_permutation(x, n_segments=5)
    return x

print('Augmentation functions defined.')

Augmentation functions defined.


In [11]:
NUM_AUG_COPIES = 2

X_aug_list = [X_train]
y_aug_list = [y_train]

for copy_idx in range(NUM_AUG_COPIES):
    print(f'Generating augmented copy {copy_idx + 1}/{NUM_AUG_COPIES}...')
    X_copy = np.array([apply_augmentation(X_train[i]) for i in range(len(X_train))], dtype=np.float32)
    X_aug_list.append(X_copy)
    y_aug_list.append(y_train.copy())

X_train_aug = np.concatenate(X_aug_list, axis=0)
y_train_aug = np.concatenate(y_aug_list, axis=0)

perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print(f'\nOriginal training: {X_train.shape[0]} windows')
print(f'Augmented training: {X_train_aug.shape[0]} windows (3x)')
print(f'Test (unchanged): {X_test.shape[0]} windows')

Generating augmented copy 1/2...
Generating augmented copy 2/2...

Original training: 9909 windows
Augmented training: 29727 windows (3x)
Test (unchanged): 1018 windows


## 10. Save Everything

In [12]:
np.save(PROCESSED_DIR / 'X_train.npy', X_train_aug)
np.save(PROCESSED_DIR / 'y_train.npy', y_train_aug)
np.save(PROCESSED_DIR / 'X_test.npy', X_test)
np.save(PROCESSED_DIR / 'y_test.npy', y_test)
np.save(PROCESSED_DIR / 'X_train_original.npy', X_train)
np.save(PROCESSED_DIR / 'y_train_original.npy', y_train)

# Save all windows + user labels for LOUO cross-validation
np.save(PROCESSED_DIR / 'X_all.npy', (X_all - sensor_mean) / sensor_std)
np.save(PROCESSED_DIR / 'y_all.npy', y_all)
np.save(PROCESSED_DIR / 'users_all.npy', users_all)

preprocessing_info = {
    'sensor_columns': ALL_SENSOR_COLS,
    'raw_sensor_columns': RAW_SENSOR_COLS,
    'num_features': NUM_FEATURES,
    'num_classes': NUM_CLASSES,
    'activities': ALL_ACTIVITIES,
    'label_map': LABEL_MAP,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'overlap': OVERLAP,
    'sampling_rate': SAMPLING_RATE,
    'sensor_mean': sensor_mean,
    'sensor_std': sensor_std,
    'user_stats': user_stats,
    'test_users': TEST_USERS,
    'train_samples_original': len(X_train),
    'train_samples_augmented': len(X_train_aug),
    'test_samples': len(X_test),
    'label_threshold': LABEL_THRESHOLD,
    'num_aug_copies': NUM_AUG_COPIES,
    'per_user_normalization': True,
    'magnitude_features': True,
}

with open(PROCESSED_DIR / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print('Saved files:')
all_files = list(PROCESSED_DIR.glob('*.npy')) + list(PROCESSED_DIR.glob('*.pkl'))
for p in sorted(all_files):
    size = p.stat().st_size
    unit = 'KB' if size < 1e6 else 'MB'
    val = size / 1024 if size < 1e6 else size / (1024*1024)
    print(f'  {p.name:35s} {val:>8.1f} {unit}')

Saved files:
  X_all.npy                               42.7 MB
  X_test.npy                               4.0 MB
  X_train.npy                            116.1 MB
  X_train_original.npy                    38.7 MB
  preprocessing_info.pkl                   2.8 KB
  users_all.npy                           85.5 KB
  y_all.npy                               42.8 KB
  y_test.npy                               4.1 KB
  y_train.npy                            116.2 KB
  y_train_original.npy                    38.8 KB


In [13]:
print('\n=== Preprocessing v3 Complete ===')
print(f'Activities: {NUM_CLASSES}')
print(f'Features: {NUM_FEATURES} ({ALL_SENSOR_COLS})')
print(f'Train windows: {len(X_train_aug)} (augmented from {len(X_train)})')
print(f'Test windows: {len(X_test)}')
print(f'Per-user normalization: YES')
print(f'Magnitude features: YES')
print(f'\nLabel mapping:')
for act, idx in LABEL_MAP.items():
    print(f'  {idx:2d}: {act}')


=== Preprocessing v3 Complete ===
Activities: 11
Features: 8 (['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w', 'Acc_mag', 'Gyro_mag'])
Train windows: 29727 (augmented from 9909)
Test windows: 1018
Per-user normalization: YES
Magnitude features: YES

Label mapping:
   0: ear_rubbing
   1: forehead_rubbing
   2: hair_pulling
   3: hand_scratching
   4: hand_tapping
   5: knuckles_cracking
   6: nail_biting
   7: nape_rubbing
   8: sitting
   9: smoking
  10: standing
